# Visualisasi Injeksi Anomali Pencurian Listrik — PLN

Notebook ini menampilkan **enam serangan pencurian listrik (f1–f6)** dari Jokar et al. (2016)
yang diimplementasikan di `data/augment.py::ElectricityTheftAttack`, diterapkan pada
sampel load profile PLN.

| Kode | Nama | Rumus |
|------|------|---------|
| f1 | Flat Reduction | $R_c(t) = \\alpha \\cdot E_c(t)$ |
| f2 | Dynamic Reduction | $R_c(t) = \\beta(t) \\cdot E_c(t)$ |
| f3 | Selective Bypass | $R_c(t) = 0$ untuk $t \\in [t_i, t_f]$, $E_c(t)$ di luar interval |
| f4 | Average Consumption | $R_c(t) = \\bar{E}_c$ |
| f5 | Dynamic Average | $R_c(t) = \\beta(t) \\cdot \\bar{E}_c$ |
| f6 | Load Reordering | $R_c(t) = E_c(T - t + 1)$ |

> **Referensi:** Jokar, P., Arianpoo, N., & Leung, V. C. M. (2016). *Electricity theft detection in AMI using customers' consumption patterns.* IEEE TSG, 7(1), 216–226.

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# ---- Resolusi CARLA root ----
def _find_carla_root():
    p = Path(os.path.abspath(''))
    for _ in range(6):
        if (p / 'carla_pretext.py').exists():
            return p
        p = p.parent
    raise RuntimeError('CARLA root tidak ditemukan.')

BASE_DIR = _find_carla_root()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
os.chdir(BASE_DIR)

from data.augment import ElectricityTheftAttack

DATA_DIR = BASE_DIR / 'datasets' / 'PLN' / 'processed' / 'data'
FIG_DIR  = BASE_DIR / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'BASE_DIR : {BASE_DIR}')
print(f'DATA_DIR : {DATA_DIR}')

## 1. Pilih Sampel File & Jendela

In [ ]:
# ============================================================
# KONFIGURASI — sesuaikan di sini
# ============================================================
WINDOW_SIZE = 96        # 96 × 15 menit = 1 hari
FEATURE_IDX = 0         # fitur (kolom) yang divisualisasikan; None = sum semua kolom
SEED        = 42
ALPHA       = 0.4       # contoh alpha untuk f1
BETA_FIXED  = 0.4       # contoh beta untuk f2 & f5
UNIT        = 'kWh'     # label satuan sumbu y — sesuaikan jika berbeda (kW, kVAr, A, ...)

# ---- Pilih file normal dari train_list.txt ----
train_list_path = BASE_DIR / 'datasets' / 'PLN' / 'processed' / 'train_list.txt'
if train_list_path.exists():
    with open(train_list_path) as f:
        train_files = [l.strip() for l in f if l.strip()]
    SAMPLE_FILE = train_files[0]
else:
    SAMPLE_FILE = sorted(DATA_DIR.glob('*-n.npy'))[0].name

raw = np.load(DATA_DIR / SAMPLE_FILE).astype(np.float32)
print(f'File     : {SAMPLE_FILE}')
print(f'Shape    : {raw.shape}  (T={raw.shape[0]}, fitur={raw.shape[1] if raw.ndim>1 else 1})')

# ---- Ambil satu jendela dari tengah rekaman ----
T = raw.shape[0]
mid = max(0, T // 2 - WINDOW_SIZE // 2)
window_raw = raw[mid : mid + WINDOW_SIZE]   # (96, 8)

if window_raw.ndim == 1:
    window_raw = window_raw[:, None]

# ---- Statistik per-file (untuk normalisasi model — disimpan tapi BUKAN untuk sumbu y) ----
mean_f = raw.mean(axis=0)
std_f  = np.where(raw.std(axis=0) == 0, 1.0, raw.std(axis=0))
raw_norm = (raw - mean_f) / (std_f + 1e-8)
window_norm = raw_norm[mid : mid + WINDOW_SIZE]
if window_norm.ndim == 1:
    window_norm = window_norm[:, None]

# ---- Kolom yang divisualisasikan ----
n_features = window_raw.shape[1]
if FEATURE_IDX is None:
    col_raw  = window_raw.sum(axis=1)
    col_label = f'Sum semua fitur ({UNIT})'
else:
    fidx     = min(FEATURE_IDX, n_features - 1)
    col_raw  = window_raw[:, fidx]
    col_label = f'Fitur {fidx} ({UNIT})'

print(f'Window   : timestep [{mid}:{mid+WINDOW_SIZE}] (hari ke-{mid//WINDOW_SIZE+1})')
print(f'Kolom    : {col_label}  |  n_features={n_features}')
print(f'Nilai raw: min={col_raw.min():.2f}  max={col_raw.max():.2f}  mean={col_raw.mean():.2f}')


## 2. Terapkan Serangan f1–f6

In [ ]:
torch.manual_seed(SEED)

# Buat instance dengan alpha/beta fixed untuk ilustrasi yang reproducible
attacker = ElectricityTheftAttack(
    alpha_range=(ALPHA, ALPHA + 1e-6),
    beta_range=(BETA_FIXED, BETA_FIXED + 1e-6)
)

# Attack diterapkan pada data RAW (kWh asli) — sesuai order fisik Jokar 2016
# attacks menyimpan nilai kWh SETELAH serangan (belum dinormalisasi)
col_t_raw = torch.tensor(col_raw, dtype=torch.float32)

attacks = {
    'f1 — Flat Reduction\n' + r'$R_c(t)=\alpha \cdot E_c(t)$' + f'  [α={ALPHA}]':
        attacker._f1_flat_reduction(col_t_raw).numpy(),

    'f2 — Dynamic Reduction\n' + r'$R_c(t)=\beta(t) \cdot E_c(t)$' + f'  [β∈({BETA_FIXED:.1f})]':
        attacker._f2_dynamic_reduction(col_t_raw).numpy(),

    'f3 — Selective Bypass\n' + r'$R_c(t)=0$ untuk $t\in[t_i,t_f]$':
        attacker._f3_selective_bypass(col_t_raw).numpy(),

    'f4 — Average Consumption\n' + r'$R_c(t)=\bar{E}_c$':
        attacker._f4_average(col_t_raw).numpy(),

    'f5 — Dynamic Average\n' + r'$R_c(t)=\beta(t)\cdot\bar{E}_c$':
        attacker._f5_dynamic_average(col_t_raw).numpy(),

    'f6 — Load Reordering\n' + r'$R_c(t)=E_c(T-t+1)$':
        attacker._f6_load_reorder(col_t_raw).numpy(),
}

# Selisih dalam satuan kWh asli (lebih interpretable)
print(f'{"Serangan":<35}  {"MAE ("+UNIT+")":>12}  {"MSE":>10}  {"Reduksi rata-rata":>18}')
print('-' * 80)
for name, attacked in attacks.items():
    mae  = np.mean(np.abs(attacked - col_raw))
    mse  = np.mean((attacked - col_raw)**2)
    red  = (col_raw.mean() - attacked.mean()) / max(col_raw.mean(), 1e-9) * 100
    print(f"{name.split(chr(10))[0]:<35}  {mae:>12.2f}  {mse:>10.2f}  {red:>17.1f}%")


## 3. Gambar — Sampel Window 1 Hari

In [ ]:
# Waktu sumbu x dalam jam (96 slot × 15 menit)
t_hours = np.arange(WINDOW_SIZE) * 15 / 60   # 0 .. 24 jam

COLORS = {
    'original': '#1a1a2e',
    'f1': '#e63946',
    'f2': '#f4a261',
    'f3': '#2a9d8f',
    'f4': '#457b9d',
    'f5': '#8338ec',
    'f6': '#c77dff',
}
attack_colors = list(COLORS.values())[1:]

fig = plt.figure(figsize=(18, 13))
fig.suptitle(
    f'Injeksi Anomali Pencurian Listrik f1–f6 — {SAMPLE_FILE}\n'
    f'(window hari ke-{mid//WINDOW_SIZE+1}, {col_label})',
    fontsize=14, fontweight='bold', y=0.98
)

# ---- Layout: baris atas = panel besar original, baris bawah = 3+3 serangan ----
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

# Panel 0: original (lebar penuh baris pertama)
ax_orig = fig.add_subplot(gs[0, :])
ax_orig.plot(t_hours, col_raw, color=COLORS['original'], linewidth=2)
ax_orig.fill_between(t_hours, col_raw, alpha=0.15, color=COLORS['original'])
ax_orig.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax_orig.set_title('ORIGINAL — Load Profile Normal $E_c(t)$', fontsize=12, fontweight='bold')
ax_orig.set_xlabel('Waktu (jam)')
ax_orig.set_ylabel(col_label)
ax_orig.set_xlim(0, 24)
ax_orig.set_xticks(range(0, 25, 4))
ax_orig.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 4)])

# Panel f1–f6: 2 baris × 3 kolom
attack_names_ordered = list(attacks.keys())
attack_vals_ordered  = list(attacks.values())

for idx in range(6):
    row = 1 + idx // 3
    col = idx % 3
    ax = fig.add_subplot(gs[row, col])

    color = attack_colors[idx]
    name  = attack_names_ordered[idx]
    val   = attack_vals_ordered[idx]   # nilai raw kWh setelah serangan

    # Shading area perbedaan dari original
    ax.fill_between(t_hours, col_raw, val,
                    where=(val < col_raw), color=color, alpha=0.25, label='Pengurangan')
    ax.fill_between(t_hours, col_raw, val,
                    where=(val >= col_raw), color='#999999', alpha=0.15, label='Kelebihan')
    ax.plot(t_hours, col_raw, color=COLORS['original'], linewidth=1,
            alpha=0.45, linestyle='--', label='Asli')
    ax.plot(t_hours, val, color=color, linewidth=1.8, label='Diserang')

    ax.axhline(0, color='gray', linestyle=':', linewidth=0.6)
    ax.set_title(name, fontsize=9.5, fontweight='bold', color=color, pad=4)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 6))
    ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 6)], fontsize=7.5)
    ax.set_xlabel('Waktu (jam)', fontsize=8)
    ax.set_ylabel(UNIT, fontsize=8)
    ax.tick_params(labelsize=7.5)

    mae = np.mean(np.abs(val - col_raw))
    red = (col_raw.mean() - val.mean()) / max(col_raw.mean(), 1e-9) * 100
    ax.text(0.98, 0.04, f'MAE={mae:.1f} {UNIT}  ({red:.0f}% reduksi)',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=7, color='gray')
    ax.legend(loc='upper right', fontsize=6.5, framealpha=0.7)

out_path = FIG_DIR / f'theft_attacks_window_{SAMPLE_FILE.replace(".npy","")}_{FEATURE_IDX}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')


## 4. Gambar — Profil Beban Penuh (Seluruh Rekaman)

In [ ]:
torch.manual_seed(SEED)

# Profil beban penuh — semua data raw kWh dari file
_fi = fidx if FEATURE_IDX is not None else 0
col_full_raw = raw[:, _fi]   # sumbu y: kWh asli

full_t = torch.tensor(col_full_raw, dtype=torch.float32)

attacker_full = ElectricityTheftAttack(
    alpha_range=(ALPHA, ALPHA + 1e-6),
    beta_range=(BETA_FIXED, BETA_FIXED + 1e-6)
)

# Nilai raw kWh setelah masing-masing serangan
full_attacks = {
    'f1': attacker_full._f1_flat_reduction(full_t).numpy(),
    'f2': attacker_full._f2_dynamic_reduction(full_t).numpy(),
    'f3': attacker_full._f3_selective_bypass(full_t).numpy(),
    'f4': attacker_full._f4_average(full_t).numpy(),
    'f5': attacker_full._f5_dynamic_average(full_t).numpy(),
    'f6': attacker_full._f6_load_reorder(full_t).numpy(),
}

full_attack_labels = [
    'f1 — Flat Reduction',
    'f2 — Dynamic Reduction',
    'f3 — Selective Bypass',
    'f4 — Average Consumption',
    'f5 — Dynamic Average',
    'f6 — Load Reordering',
]

t_days = np.arange(len(col_full_raw)) / WINDOW_SIZE   # sumbu x dalam hari

fig, axes = plt.subplots(7, 1, figsize=(18, 22), sharex=True)
fig.suptitle(
    f'Profil Beban Penuh dengan Injeksi f1–f6 — {SAMPLE_FILE}\n'
    f'(Fitur {_fi}, satuan {UNIT}, nilai asli)',
    fontsize=13, fontweight='bold', y=0.995
)

# Panel original
axes[0].plot(t_days, col_full_raw, color=COLORS['original'], linewidth=0.9)
axes[0].fill_between(t_days, col_full_raw, alpha=0.12, color=COLORS['original'])
axes[0].set_title('ORIGINAL $E_c(t)$', fontsize=10, fontweight='bold')
axes[0].set_ylabel(UNIT, fontsize=8)

for idx, (key, val) in enumerate(full_attacks.items()):
    ax = axes[idx + 1]
    color = attack_colors[idx]
    label = full_attack_labels[idx]

    ax.fill_between(t_days, col_full_raw, val,
                    where=(val < col_full_raw), color=color, alpha=0.3)
    ax.plot(t_days, col_full_raw, color=COLORS['original'], linewidth=0.7,
            alpha=0.4, linestyle='--')
    ax.plot(t_days, val, color=color, linewidth=1.0)

    mae = np.mean(np.abs(val - col_full_raw))
    red = (col_full_raw.mean() - val.mean()) / max(col_full_raw.mean(), 1e-9) * 100
    ax.set_title(f'{label}  (MAE={mae:.1f} {UNIT}, reduksi={red:.0f}%)',
                 fontsize=9.5, fontweight='bold', color=color)
    ax.set_ylabel(UNIT, fontsize=8)
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.5)
    ax.tick_params(labelsize=7.5)

axes[-1].set_xlabel('Hari ke-', fontsize=10)
for ax in axes:
    ax.set_xlim(0, t_days.max())
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.995])
out_path2 = FIG_DIR / f'theft_attacks_fullprofile_{SAMPLE_FILE.replace(".npy","")}_{_fi}.png'
plt.savefig(out_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path2}')


## 5. Gambar — Semua 8 Fitur dalam Satu Window

In [ ]:
torch.manual_seed(SEED)

n_feat   = window_raw.shape[1]
n_attack = 6
attack_fns = [
    ('f1', attacker._f1_flat_reduction),
    ('f2', attacker._f2_dynamic_reduction),
    ('f3', attacker._f3_selective_bypass),
    ('f4', attacker._f4_average),
    ('f5', attacker._f5_dynamic_average),
    ('f6', attacker._f6_load_reorder),
]
attack_short_labels = [
    'f1 Flat Reduction',
    'f2 Dynamic Reduction',
    'f3 Selective Bypass',
    'f4 Average',
    'f5 Dynamic Average',
    'f6 Load Reordering',
]

fig, axes = plt.subplots(n_feat, n_attack + 1, figsize=(22, 2.4 * n_feat),
                          sharey='row', sharex=True)
fig.suptitle(
    f'Semua Fitur × Semua Serangan — {SAMPLE_FILE}\n'
    f'(window hari ke-{mid//WINDOW_SIZE+1}, nilai {UNIT} asli)',
    fontsize=12, fontweight='bold', y=1.01
)

for fi in range(n_feat):
    raw_col = window_raw[:, fi]                               # kWh asli fitur fi
    col_t_fi = torch.tensor(raw_col, dtype=torch.float32)

    # Kolom 0: original
    ax0 = axes[fi, 0]
    ax0.plot(t_hours, raw_col, color=COLORS['original'], linewidth=1.3)
    ax0.fill_between(t_hours, raw_col, alpha=0.15, color=COLORS['original'])
    ax0.set_ylabel(f'F{fi}\n({UNIT})', fontsize=7.5, fontweight='bold')
    ax0.tick_params(labelsize=6)
    if fi == 0:
        ax0.set_title('Original', fontsize=9, fontweight='bold')

    # Kolom 1–6: serangan (nilai raw kWh)
    for ai, (acode, afn) in enumerate(attack_fns):
        attacked = afn(col_t_fi).numpy()          # raw kWh setelah serangan
        color = attack_colors[ai]
        ax = axes[fi, ai + 1]
        ax.plot(t_hours, raw_col, color=COLORS['original'], linewidth=0.8,
                alpha=0.35, linestyle='--')
        ax.plot(t_hours, attacked, color=color, linewidth=1.3)
        ax.fill_between(t_hours, raw_col, attacked,
                        where=(attacked < raw_col), color=color, alpha=0.2)
        ax.tick_params(labelsize=6)
        if fi == 0:
            ax.set_title(attack_short_labels[ai], fontsize=8.5, fontweight='bold', color=color)

# Sumbu x: hanya baris bawah
for ai in range(n_attack + 1):
    axes[-1, ai].set_xticks(range(0, 25, 6))
    axes[-1, ai].set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 6)], fontsize=6.5, rotation=30)
    axes[-1, ai].set_xlabel('Jam', fontsize=7)

for ax_row in axes:
    for ax in ax_row:
        ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out_path3 = FIG_DIR / f'theft_attacks_allfeatures_{SAMPLE_FILE.replace(".npy","")}.png'
plt.savefig(out_path3, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path3}')


## 6. Gambar — Overlay f1–f6 dalam Satu Panel

In [ ]:
torch.manual_seed(SEED)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(t_hours, col_raw, color=COLORS['original'], linewidth=2.5,
        label=f'Original $E_c(t)$  (mean={col_raw.mean():.1f} {UNIT})', zorder=10)

for idx, (name, val) in enumerate(attacks.items()):
    short = name.split('\n')[0]
    red = (col_raw.mean() - val.mean()) / max(col_raw.mean(), 1e-9) * 100
    ax.plot(t_hours, val, color=attack_colors[idx], linewidth=1.5,
            linestyle='-', alpha=0.85, label=f'{short}  (−{red:.0f}%)')

ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Waktu (jam)', fontsize=11)
ax.set_ylabel(col_label, fontsize=11)
ax.set_title(
    f'Overlay Semua Serangan — {SAMPLE_FILE}',
    fontsize=12, fontweight='bold'
)
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 2)], rotation=45, fontsize=8)
ax.legend(loc='upper left', fontsize=8.5, framealpha=0.85, ncol=2)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out_path4 = FIG_DIR / f'theft_attacks_overlay_{SAMPLE_FILE.replace(".npy","")}.png'
plt.savefig(out_path4, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path4}')


## 7. Gambar — Alpha Sweep untuk f1 (Flat Reduction)

In [ ]:
# Alpha sweep untuk f1 — nilai kWh asli
alphas = [0.1, 0.2, 0.4, 0.6, 0.8]
cmap   = plt.cm.Reds

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(t_hours, col_raw, color='black', linewidth=2.5,
        label=f'Original $E_c(t)$  (mean={col_raw.mean():.1f} {UNIT})', zorder=10)

for a in alphas:
    att = ElectricityTheftAttack(alpha_range=(a, a + 1e-9))
    y = att._f1_flat_reduction(col_t_raw).numpy()   # raw kWh (= alpha * Ec)
    mean_y = y.mean()
    ax.plot(t_hours, y, linewidth=1.5, alpha=0.85,
            color=cmap(0.2 + 0.7 * a), label=f'f1  α={a}  (mean={mean_y:.1f} {UNIT})')

ax.fill_between(t_hours, col_raw, alpha=0.08, color='black')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Waktu (jam)', fontsize=11)
ax.set_ylabel(col_label, fontsize=11)
ax.set_title(f'f1 — Flat Reduction: Pengaruh nilai α (0.1–0.8)  [{SAMPLE_FILE}]',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 2)], rotation=45, fontsize=8)
ax.legend(fontsize=9, framealpha=0.85)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out_path5 = FIG_DIR / f'theft_f1_alpha_sweep_{SAMPLE_FILE.replace(".npy","")}.png'
plt.savefig(out_path5, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path5}')


## 8. Ringkasan

File-file gambar yang dihasilkan:

| Gambar | Isi |
|--------|-----|
| `theft_attacks_window_*.png` | Layout 2×3: original atas + f1–f6 bawah, satu window (1 hari) |
| `theft_attacks_fullprofile_*.png` | 7 panel vertikal: profil beban penuh semua hari |
| `theft_attacks_allfeatures_*.png` | Grid fitur × serangan: semua 8 fitur × 7 kolom |
| `theft_attacks_overlay_*.png` | Overlay semua serangan dalam satu panel |
| `theft_f1_alpha_sweep_*.png` | Pengaruh berbagai nilai α pada f1 |

> **Catatan:** Gambar menggunakan data yang sudah **ternormalisasi per-file** (sama seperti yang diterima model CARLA). Sumbu Y adalah konsumsi dalam satuan standar deviasi dari rata-rata file tersebut.